In [126]:
using LowLevelFEM, LinearAlgebra

In [127]:
openGeometry("boxes.geo")

In [128]:
#openPreProcessor()

In [129]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [130]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=false)

  0.527427 seconds (115.75 k allocations: 50.601 MiB, 1.89% gc time, 26.01% compilation time)


0

In [131]:
contact_pair = contact(u, master="master", slave="slave", cn=1e8)

Contact("slave" -> "master", 788 candidate nodes, 402 active, G=(2364, 9438), C=(2364, 2364))

In [132]:
using SparseArrays

# Lagrange multiplier field
Λ = Field([mat], type=:VectorField, dim=3, fieldName=:λ)

# Contact kinematics
L = contact(
    u,
    master="master",
    slave="slave",
    LagrangeMultiplierField=Λ
)

support = [bc_bottom, bc_top]
free_u = freeDoFs(U, support)

u_it = copy(u)
λ_it = vectorField(Λ, "body", [0, 0, 0])

# Zero multiplier block
nλ = size(L.E, 1)
Zλ = SystemMatrix(spzeros(nλ, nλ), Λ)

# Primal-dual active-set parameter.
# This is NOT a penalty stiffness; it is used only to determine the active set.
κ = 1e8

active_old = falses(length(L.slave_nodes))

for iter in 1:40

    # Current contact geometry
    updateContact!(L, u_it)

    (; G, g, E) = L

    pdim = L.U.pdim

    # Normal component in the reduced contact space
    normal_rows = 1:pdim:length(g)

    # Corresponding normal multiplier DoFs
    λn_dofs = L.multiplier_dofs[normal_rows]
    λn = DoFs(λ_it)[λn_dofs]

    # ----------------------------------------------------------
    # Primal-dual active set
    #
    # Sign convention:
    #     g_n >= 0       open/admissible
    #     λ_n <= 0       compression
    #
    # active <=> λ_n + κ g_n < 0
    # ----------------------------------------------------------
    active = λn .+ κ .* L.gap_values .< 0.0

    # Inactive multipliers are zero
    DoFs(λ_it)[λn_dofs[.!active]] .= 0.0

    # ----------------------------------------------------------
    # Active normal projector in contact space
    # ----------------------------------------------------------
    χ = zeros(Float64, length(g))
    χ[normal_rows[active]] .= 1.0

    P = SystemMatrix(
        spdiagm(0 => χ),
        nothing,
        nothing,
        nothing,
        nothing
    )

    # ----------------------------------------------------------
    # Contact constraint operator
    #
    #     B = E P G
    #     gλ = E P g
    # ----------------------------------------------------------
    B  = E * P * G
    gλ = E * P * g

    # ----------------------------------------------------------
    # KKT residual
    #
    #     rᵤ = K u - f + B' λ
    #     rλ = gλ
    # ----------------------------------------------------------
    rᵤ = K * u_it - f + B' * λ_it
    rλ = gλ

    # ----------------------------------------------------------
    # KKT tangent
    #
    #         [ K   B' ]
    #     A = [        ]
    #         [ B    0 ]
    # ----------------------------------------------------------
    A = SystemMatrix([
        K   B'
        B   Zλ
    ])

    r = SystemVector([rᵤ, rλ])

    # Offset of the multiplier field in the multifield system
    λoff = A.offsets[2]

    # Only displacement free DoFs and ACTIVE NORMAL multiplier DoFs
    # participate in the Newton correction.
    free = vcat(
        free_u,
        λoff .+ λn_dofs[active]
    )

    Δx = zeros(Float64, size(A, 1))

    Δx[free] =
        -A[free, free] \ r.a[free, 1]

    # Split the multifield correction
    Δu = @view Δx[1:λoff]
    Δλ = @view Δx[λoff+1:end]

    # Newton update
    DoFs(u_it)[:] .+= Δu
    DoFs(λ_it)[:] .+= Δλ

    # ----------------------------------------------------------
    # Convergence diagnostics
    # ----------------------------------------------------------
    Δactive = count(active .!= active_old)

    err_u =
        norm(Δu[free_u]) /
        max(norm(DoFs(u_it)[free_u]), eps())

    max_gap =
        any(active) ?
        maximum(abs.(L.gap_values[active])) :
        0.0

    λn = DoFs(λ_it)[λn_dofs]

    min_λ =
        any(active) ?
        minimum(λn[active]) :
        0.0

    println(
        "iter = ", iter,
        ", active = ", count(active),
        ", Δactive = ", Δactive,
        ", max |gap| = ", max_gap,
        ", min λn = ", min_λ,
        ", error = ", err_u
    )

    converged =
        Δactive == 0 &&
        max_gap < 1e-8 &&
        err_u < 1e-8

    active_old .= active

    converged && break
end

u_LM = u_it
λ_LM = λ_it

iter = 1, active = 402, Δactive = 402, max |gap| = 0.17743021201991346, min λn = -1864.2161922920584, error = 0.30990876977737297
iter = 2, active = 217, Δactive = 185, max |gap| = 4.3296431341849086e-5, min λn = -4124.108511098792, error = 0.0451270403566413
iter = 3, active = 325, Δactive = 186, max |gap| = 0.050268719311621794, min λn = -2496.738746259442, error = 0.046526446408178296
iter = 4, active = 267, Δactive = 126, max |gap| = 0.022948558852205807, min λn = -2755.312307729502, error = 0.024730236541585262
iter = 5, active = 316, Δactive = 103, max |gap| = 0.02220436751647731, min λn = -2453.8735280846895, error = 0.02203593480064774
iter = 6, active = 290, Δactive = 64, max |gap| = 0.017402645410020193, min λn = -2836.8504955115673, error = 0.017085684531245194
iter = 7, active = 317, Δactive = 55, max |gap| = 0.02364339099003418, min λn = -2101.142016959903, error = 0.016050975961301565
iter = 8, active = 308, Δactive = 35, max |gap| = 0.01807215523315658, min λn = -2708.19

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [133]:
showDoFResults(u_LM, name="u cont.", visible=true, factor=1)


1

In [134]:
λ_LM = nodesToElements(λ_LM, onPhysicalGroup="slave")
showElementResults(λ_LM[1], name="λ")

2

In [135]:
λ_LM.type

:v3D

In [136]:
showElementResults(contact_pair.gap, name="gap")

3

In [138]:
openPostProcessor()

Két vagy több párnál majd:

```Julia
contacts = ContactSet(c1, c2, c3)

updateContact!(contacts, u_it)

Kc = sum(c.G' * c.C * c.G for c in contacts)
rc = sum(c.G' * c.C * c.g for c in contacts)

r = K * u_it - f + rc
A = K + Kc
```